In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

# Cài đặt sentence-transformers cho embedding và chromadb cho vector database
!pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 774.7 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [3]:
import os
import json
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# 1. Định nghĩa các đường dẫn chuẩn
ROOT_DIR = "/content/drive/MyDrive/nutrition_rag"
PROCESSED_DIR = os.path.join(ROOT_DIR, "data_processed")
VECTOR_DB_DIR = os.path.join(ROOT_DIR, "vector_db")
ALL_RECORDS_FILE = os.path.join(PROCESSED_DIR, "all_records.json")

# 2. Đọc file Knowledge Base phục vụ cho việc lấy metadata và build BM25
with open(ALL_RECORDS_FILE, "r", encoding="utf-8") as f:
    all_records = json.load(f)
print(f"Đã load {len(all_records)} records từ all_records.json")

# Mảng documents và ids vẫn cần nạp vào RAM để Cell 5 dùng cho BM25
ids = [record["record_id"] for record in all_records]
documents = [record["text_for_embedding"] for record in all_records]

# 3. Khởi tạo Embedding Model (Dùng lại cho việc encode Query lúc sau)
print("Đang tải model BAAI/bge-m3...")
model = SentenceTransformer("BAAI/bge-m3")

# 4. Kết nối ChromaDB Persistent Client
print("Khởi tạo/Kết nối ChromaDB...")
client = chromadb.PersistentClient(path=VECTOR_DB_DIR)

collection = client.get_or_create_collection(
    name="nutrition_collection",
    metadata={"hnsw:space": "cosine"}
)

# 5. KIỂM TRA TRẠNG THÁI VECTOR DB (CHỐNG RE-EMBEDDING)
existing_count = collection.count()
if existing_count > 0:
    print("\n" + "="*50)
    print(f"TRẠNG THÁI: ChromaDB đã có sẵn {existing_count} bản ghi.")
    print("-> BỎ QUA bước tạo embedding để tiết kiệm thời gian và tài nguyên.")
    print("="*50)
else:
    print("\nTRẠNG THÁI: ChromaDB trống. Bắt đầu tạo embedding ban đầu...")
    metadatas = []
    for record in all_records:
        meta = {
            "record_type": record.get("record_type", ""),
            "source": record.get("source", "")
        }
        if record.get("group"): meta["group"] = record["group"]
        if record.get("age"): meta["age"] = record["age"]
        if record.get("food_name_en"): meta["food_name_en"] = record["food_name_en"]
        metadatas.append(meta)

    BATCH_SIZE = 100
    for i in tqdm(range(0, len(ids), BATCH_SIZE)):
        batch_ids = ids[i : i + BATCH_SIZE]
        batch_docs = documents[i : i + BATCH_SIZE]
        batch_metas = metadatas[i : i + BATCH_SIZE]

        batch_embeddings = model.encode(batch_docs, convert_to_tensor=False).tolist()

        collection.upsert(
            ids=batch_ids,
            documents=batch_docs,
            embeddings=batch_embeddings,
            metadatas=batch_metas
        )
    print(f"\nTHÀNH CÔNG! Đã khởi tạo mới {collection.count()} records vào ChromaDB.")

Đã load 583 records từ all_records.json
Đang tải model BAAI/bge-m3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Khởi tạo/Kết nối ChromaDB...

TRẠNG THÁI: ChromaDB đã có sẵn 583 bản ghi.
-> BỎ QUA bước tạo embedding để tiết kiệm thời gian và tài nguyên.


In [4]:
# Cell 3: Viết hàm Retriever và Test truy xuất

def retrieve_info(query, top_k=3):
    print(f"\n" + "="*60)
    print(f"[🔍 QUERY] {query}")
    print("="*60)

    # 1. Chuyển đổi câu hỏi thành vector (Embedding)
    query_embedding = model.encode(query, convert_to_tensor=False).tolist()

    # 2. Truy vấn ChromaDB (tìm top_k vector gần nhất)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    # 3. Hiển thị kết quả
    if not results['documents'][0]:
        print("Không tìm thấy kết quả phù hợp.")
        return

    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        dist = results['distances'][0][i] # Cosine distance (càng nhỏ càng giống)

        print(f"\n--- KẾT QUẢ {i+1} (Khoảng cách: {dist:.4f}) ---")
        print(f"Loại Record: {meta.get('record_type', 'N/A').upper()}")
        if 'food_name_en' in meta:
            print(f"Tên (EN): {meta['food_name_en']}")
        if 'group' in meta and 'age' in meta:
            print(f"Đối tượng: {meta['group']} | Tuổi: {meta['age']}")
        print(f"Trích xuất:\n{doc}")

# Chạy 3 test case theo đúng kế hoạch
test_queries = [
    "How much protein is in beef?",
    "Calcium requirement for adult male",
    "Vitamin C recommendation for children"
]

for q in test_queries:
    # Lấy top 2 kết quả tốt nhất cho dễ nhìn
    retrieve_info(q, top_k=2)


[🔍 QUERY] How much protein is in beef?

--- KẾT QUẢ 1 (Khoảng cách: 0.3229) ---
Loại Record: FOOD
Tên (EN): Beef, blood
Trích xuất:
Food name: Beef, blood
Calories: 75.0 kcal
Protein: 18.0 g
Fat: 0.2 g
Carbohydrate: 0.4 g
Fiber: 0.0 g
Calcium: 8.0 mg
Iron: 52.6 mg

--- KẾT QUẢ 2 (Khoảng cách: 0.3305) ---
Loại Record: FOOD
Tên (EN): Beef, grade I
Trích xuất:
Food name: Beef, grade I
Calories: 118.0 kcal
Protein: 21.0 g
Fat: 3.8 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 12.0 mg
Iron: 3.1 mg

[🔍 QUERY] Calcium requirement for adult male

--- KẾT QUẢ 1 (Khoảng cách: 0.2614) ---
Loại Record: MINERAL
Đối tượng: Nam trưởng thành | Tuổi: 19-49 tuổi
Trích xuất:
Group: Nam trưởng thành
Age: 19-49 tuổi
Calcium: 700 mg
Magnesium: 205 mg
Phosphorus: 700 mg
Selenium: 34 mcg

--- KẾT QUẢ 2 (Khoảng cách: 0.2630) ---
Loại Record: MINERAL
Đối tượng: Nam trưởng thành | Tuổi: 50-60 tuổi
Trích xuất:
Group: Nam trưởng thành
Age: 50-60 tuổi
Calcium: 1000 mg
Magnesium: 205 mg
Phosphorus: 700 mg
Selenium: 3

In [5]:
# Cài đặt thư viện Google Generative AI
!pip install -q google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Lấy API Key từ Secrets của Colab
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError:
    print("LỖI: Vui lòng thêm GEMINI_API_KEY vào tab Secrets của Colab!")

# Sử dụng model gemini-1.5-flash (nhanh, rẻ và đủ thông minh cho tác vụ RAG)
llm = genai.GenerativeModel("gemini-2.5-flash")

def build_context(query, top_k=4):
    """Truy xuất thông tin từ ChromaDB và gộp thành chuỗi văn bản (Context)"""
    query_embedding = model.encode(query, convert_to_tensor=False).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    if not results['documents'][0]:
        return ""

    context_str = ""
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        source = results['metadatas'][0][i].get('source', 'Unknown database')
        context_str += f"--- Bản ghi {i+1} (Nguồn: {source}) ---\n{doc}\n\n"

    return context_str

def ask_nutrition_rag(query):
    """Hàm Pipeline RAG hoàn chỉnh từ người dùng -> ChromaDB -> Gemini -> Phản hồi"""
    print(f"👤 Câu hỏi: {query}\n")

    # 1. Retrieval (Truy xuất)
    context = build_context(query, top_k=4)

    if not context:
        print("🤖 Trả lời: Xin lỗi, tôi không tìm thấy thông tin phù hợp trong dữ liệu hiện có.")
        print("="*60)
        return

    # 2. Prompt Engineering (Thiết kế lời nhắc chặt chẽ chống ảo giác)
    prompt = f"""Bạn là một chuyên gia dinh dưỡng AI hỗ trợ người dùng tra cứu thông tin.

    QUY TẮC BẮT BUỘC:
    1. CHỈ sử dụng thông tin từ phần [CONTEXT] bên dưới. TUYỆT ĐỐI không tự bịa số liệu.
    2. Các con số (calo, protein, vitamin...) PHẢI được trích dẫn kèm theo đơn vị cụ thể.
    3. Nếu ngữ cảnh [CONTEXT] không đủ thông tin để trả lời, hãy nói thẳng: "Tôi không tìm thấy thông tin này trong tài liệu hiện có."
    4. Trả lời ngắn gọn, trực tiếp, và mạch lạc bằng tiếng Việt.

    [CONTEXT]
    {context}

    [CÂU HỎI CỦA NGƯỜI DÙNG]
    {query}

    [CÂU TRẢ LỜI CỦA BẠN]
    """

    # 3. Generation (Sinh câu trả lời)
    response = llm.generate_content(prompt)

    print(f"🤖 Trả lời:\n{response.text.strip()}")
    print("\n" + "="*80 + "\n")

# Chạy thử RAG End-to-End với các câu hỏi thực tế
ask_nutrition_rag("Thịt bò chứa khoảng bao nhiêu protein?")
ask_nutrition_rag("Nam giới 35 tuổi cần bổ sung bao nhiêu canxi mỗi ngày?")
ask_nutrition_rag("Trẻ em 5 tuổi cần cung cấp bao nhiêu Vitamin C theo khuyến nghị?")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


👤 Câu hỏi: Thịt bò chứa khoảng bao nhiêu protein?

🤖 Trả lời:
Trong tài liệu hiện có, tiết bò (Beef, blood) chứa 18.0 g protein.


👤 Câu hỏi: Nam giới 35 tuổi cần bổ sung bao nhiêu canxi mỗi ngày?

🤖 Trả lời:
Nam giới 35 tuổi (thuộc nhóm Nam trưởng thành, 19-49 tuổi) cần bổ sung 700 mg canxi mỗi ngày.


👤 Câu hỏi: Trẻ em 5 tuổi cần cung cấp bao nhiêu Vitamin C theo khuyến nghị?

🤖 Trả lời:
Theo khuyến nghị, trẻ em 5 tuổi cần cung cấp 30.0 mg Vitamin C.




In [6]:
# Cài đặt thư viện rank_bm25
!pip install -q rank_bm25

from rank_bm25 import BM25Okapi
import numpy as np

print("Đang khởi tạo BM25 Index...")
# Biến 'documents' và 'ids' đã được khai báo ở Cell 2
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("Khởi tạo BM25 hoàn tất!")

def reciprocal_rank_fusion(dense_results, sparse_results, k=60):
    """Kết hợp kết quả dense và sparse search theo công thức RRF"""
    scores = {}
    for rank, doc_id in enumerate(dense_results):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    for rank, doc_id in enumerate(sparse_results):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    # Sắp xếp giảm dần theo điểm RRF
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(query, top_k=4):
    """Tìm kiếm lai kết hợp cả từ khóa chính xác và ngữ nghĩa"""
    # 1. Sparse Search (BM25) - Lấy Top 50
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    sparse_top_indices = np.argsort(bm25_scores)[::-1][:50]
    sparse_results = [ids[i] for i in sparse_top_indices]

    # 2. Dense Search (ChromaDB) - Lấy Top 50
    query_embedding = model.encode(query, convert_to_tensor=False).tolist()
    dense_response = collection.query(
        query_embeddings=[query_embedding],
        n_results=50
    )
    dense_results = dense_response['ids'][0]

    # 3. Kết hợp bằng RRF Fusion
    rrf_ranking = reciprocal_rank_fusion(dense_results, sparse_results)

    # Lấy Top K doc_ids cuối cùng
    top_k_ids = [doc_id for doc_id, score in rrf_ranking[:top_k]]

    # Kéo nội dung text từ ChromaDB dựa trên doc_ids
    final_context_str = ""
    for i, doc_id in enumerate(top_k_ids):
        # Truy vấn ngược lại ChromaDB để lấy text bằng ID
        doc_data = collection.get(ids=[doc_id])
        if doc_data and doc_data['documents']:
            text = doc_data['documents'][0]
            source = doc_data['metadatas'][0].get('source', 'Unknown')
            final_context_str += f"--- Bản ghi {i+1} (ID: {doc_id} | Nguồn: {source}) ---\n{text}\n\n"

    return final_context_str

def ask_nutrition_rag_hybrid(query):
    """Pipeline RAG ứng dụng Hybrid Search"""
    print(f"👤 Câu hỏi: {query}\n")

    context = hybrid_search(query, top_k=4)

    if not context:
        print("🤖 Trả lời: Xin lỗi, tôi không tìm thấy thông tin.")
        return

    prompt = f"""Bạn là một chuyên gia dinh dưỡng AI.
    QUY TẮC BẮT BUỘC:
    1. CHỈ sử dụng thông tin từ [CONTEXT]. Không tự bịa số liệu.
    2. Các con số phải được trích dẫn kèm đơn vị.

    [CONTEXT]
    {context}

    [CÂU HỎI CỦA NGƯỜI DÙNG]
    {query}

    [CÂU TRẢ LỜI CỦA BẠN]
    """

    response = llm.generate_content(prompt)
    print(f"🤖 Trả lời:\n{response.text.strip()}")
    print("\n" + "="*80 + "\n")

# Chạy test lại câu hỏi về thịt bò
ask_nutrition_rag_hybrid("Thịt bò (Beef, grade I) chứa khoảng bao nhiêu protein?")

Đang khởi tạo BM25 Index...
Khởi tạo BM25 hoàn tất!
👤 Câu hỏi: Thịt bò (Beef, grade I) chứa khoảng bao nhiêu protein?

🤖 Trả lời:
Thịt bò (Beef, grade I) chứa khoảng 21.0 g protein.




In [7]:
def route_query(query):
    """Sử dụng LLM để phân loại ý định câu hỏi của người dùng"""

    router_prompt = f"""
    Bạn là một bộ định tuyến hệ thống (System Router).
    Nhiệm vụ của bạn là phân loại câu hỏi của người dùng vào ĐÚNG 1 TRONG 4 loại sau:

    - LOOKUP: Tra cứu thông số dinh dưỡng cụ thể (ví dụ: calo, protein của 1 món).
    - COMPARE: So sánh thông tin giữa 2 hoặc nhiều loại thực phẩm.
    - RECOMMEND: Yêu cầu gợi ý thực đơn, chế độ ăn, hoặc lời khuyên tổng hợp.
    - EXPLAIN: Yêu cầu giải thích các khái niệm dinh dưỡng, cơ chế tác động.

    Câu hỏi: "{query}"

    Chỉ trả về 1 từ duy nhất: LOOKUP, COMPARE, RECOMMEND, hoặc EXPLAIN. Không giải thích gì thêm.
    """

    response = llm.generate_content(router_prompt)
    intent = response.text.strip().upper()

    # Xử lý trường hợp LLM sinh ra text thừa
    valid_intents = ["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"]
    for valid_intent in valid_intents:
        if valid_intent in intent:
            return valid_intent

    return "LOOKUP" # Fallback mặc định nếu không xác định được

# Dictionary cấu hình thông số linh hoạt dựa trên Intent
ROUTE_CONFIG = {
    "LOOKUP": {"top_k": 3},
    "COMPARE": {"top_k": 6},
    "RECOMMEND": {"top_k": 10},
    "EXPLAIN": {"top_k": 5}
}

# ----------------- TEST ROUTER -----------------
test_queries = [
    "1 quả táo bao nhiêu calo?",
    "Thịt bò và thịt lợn cái nào nhiều đạm hơn?",
    "Lên thực đơn 1500 kcal cho người tiểu đường",
    "Tại sao trẻ em cần bổ sung nhiều canxi?"
]

print("ĐANG TEST QUERY ROUTER...\n" + "="*50)
for q in test_queries:
    intent = route_query(q)
    config = ROUTE_CONFIG.get(intent)
    print(f"Câu hỏi: {q}")
    print(f"👉 Phân loại: {intent} | Số lượng record cần lấy (top_k): {config['top_k']}\n")

In [8]:
def final_ask_nutrition_rag(query):
    """Hệ thống RAG hoàn chỉnh kết hợp Router, Hybrid Search và LLM"""
    print(f"👤 Câu hỏi: {query}")

    # 1. Router xác định ý định và số lượng record cần lấy
    intent = route_query(query)
    top_k = ROUTE_CONFIG.get(intent, {"top_k": 4})["top_k"]
    print(f"⚙️ Phân loại: {intent} | Tiến hành Hybrid Search lấy Top {top_k} records...\n")

    # 2. Hybrid Retrieval
    context = hybrid_search(query, top_k=top_k)

    if not context:
        print("🤖 Trả lời: Xin lỗi, tôi không tìm thấy thông tin phù hợp trong cơ sở dữ liệu.")
        print("="*80)
        return

    # 3. Prompt Engineering (Tuân thủ nghiêm ngặt guardrails)
    prompt = f"""Bạn là một chuyên gia dinh dưỡng AI, hỗ trợ người dùng tra cứu thông tin về món ăn và dinh dưỡng dựa trên tài liệu khoa học được cung cấp.

    QUY TẮC BẮT BUỘC:
    1. CHỈ sử dụng thông tin từ [CONTEXT] bên dưới. TUYỆT ĐỐI không tự bịa số liệu.
    2. Mọi con số (calo, protein, vitamin...) PHẢI được trích dẫn rõ ràng.
    3. Nếu context không đủ thông tin, hãy nói thẳng: "Tôi không tìm thấy thông tin trong tài liệu hiện có."
    4. Trình bày mạch lạc, dễ hiểu. Nếu câu hỏi yêu cầu so sánh, hãy trình bày rõ sự khác biệt.

    [CONTEXT]
    {context}

    [CÂU HỎI CỦA NGƯỜI DÙNG]
    {query}

    [CÂU TRẢ LỜI CỦA BẠN]
    """

    # 4. Generation
    response = llm.generate_content(prompt)

    print(f"🤖 Trả lời:\n{response.text.strip()}")
    print("\n" + "="*80 + "\n")

# ==================== CHẠY THỬ HỆ THỐNG HOÀN CHỈNH ====================
queries = [
    "Thịt bò (Beef, grade I) và thịt lợn mỡ (Pork, fat) thì loại nào chứa nhiều calo hơn?",
    "Tôi là nữ trưởng thành 25 tuổi, tôi cần bao nhiêu Sắt và Canxi mỗi ngày?",
]

for q in queries:
    final_ask_nutrition_rag(q)

In [9]:
import os
import json
from sklearn.metrics import classification_report
import numpy as np

# Đảm bảo thư mục outputs tồn tại
OUTPUT_DIR = "/content/drive/MyDrive/nutrition_rag/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("BẮT ĐẦU ĐÁNH GIÁ HỆ THỐNG (EVALUATION FRAMEWORK)")
print("="*60)

# ==========================================
# CẤP 1: ĐÁNH GIÁ INTENT ROUTER
# ==========================================
print("\n[1] ĐANG ĐÁNH GIÁ CẤP 1 - INTENT ROUTER...")

# Tập dữ liệu test cho Router (Ground Truth)
router_test_data = [
    ("Cơm trắng 100g có bao nhiêu calo?", "LOOKUP"),
    ("Hàm lượng vitamin C trong cam là bao nhiêu?", "LOOKUP"),
    ("Thịt bò và thịt gà loại nào nhiều protein hơn?", "COMPARE"),
    ("Sữa tươi nguyên kem so với sữa ít béo khác gì?", "COMPARE"),
    ("Tôi muốn giảm cân, ăn gì cho bữa sáng?", "RECOMMEND"),
    ("Thực đơn 7 ngày cho người tiểu đường type 2", "RECOMMEND"),
    ("Chỉ số GI là gì và ảnh hưởng thế nào đến đường huyết?", "EXPLAIN"),
    ("Tại sao thiếu sắt lại gây thiếu máu?", "EXPLAIN")
]

y_true = []
y_pred = []

for query, true_intent in router_test_data:
    pred_intent = route_query(query)
    y_true.append(true_intent)
    y_pred.append(pred_intent)

# Tạo báo cáo phân loại
router_report = classification_report(
    y_true, y_pred,
    labels=["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"],
    output_dict=True,
    zero_division=0
)

# In báo cáo ra màn hình
print(classification_report(
    y_true, y_pred,
    labels=["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"],
    zero_division=0
))

# ==========================================
# CẤP 2: ĐÁNH GIÁ RETRIEVAL PIPELINE
# ==========================================
print("\n[2] ĐANG ĐÁNH GIÁ CẤP 2 - HYBRID RETRIEVAL...")

# Để đánh giá chính xác, chúng ta cần biết "record_id" chứa đáp án.
# Vì chúng ta gen id tự động (food_00xx, min_00xx), ta sẽ tạo một hàm tìm ID để làm Ground Truth ảo cho test.
def get_id_by_keyword(keyword, r_type="food"):
    for r in all_records:
        if r["record_type"] == r_type and keyword.lower() in str(r.get("food_name_en", "")).lower():
            return r["record_id"]
    return "unknown"

# Tạo Retrieval Test Set
retrieval_test_data = [
    {
        "query": "Thịt bò (Beef, grade I) chứa bao nhiêu protein?",
        "relevant_ids": [get_id_by_keyword("Beef, grade I")],
    },
    {
        "query": "Lượng Vitamin C cho trẻ em 4-6 tuổi",
        "relevant_ids": [r["record_id"] for r in all_records if r["record_type"] == "vitamin" and "4-6" in str(r.get("age", ""))],
    },
    {
        "query": "Gạo nếp (Glutinous rice, milled) có bao nhiêu calo?",
        "relevant_ids": [get_id_by_keyword("Glutinous rice, milled")],
    }
]

def recall_at_k(retrieved_ids, relevant_ids, k):
    top_k = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    return len(top_k & relevant) / len(relevant) if relevant else 0.0

def mean_reciprocal_rank(retrieved_ids, relevant_ids):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

retrieval_results = {"Recall@1": [], "Recall@3": [], "Recall@5": [], "MRR": []}

for test_case in retrieval_test_data:
    query = test_case["query"]
    relevant = test_case["relevant_ids"]

    # Chạy Hybrid Search để lấy top 10 IDs (chỉ lấy ID, không sinh text)
    tokenized_query = query.lower().split()
    sparse_scores = bm25.get_scores(tokenized_query)
    sparse_top = np.argsort(sparse_scores)[::-1][:50]
    sparse_results = [ids[i] for i in sparse_top]

    dense_response = collection.query(
        query_embeddings=[model.encode(query, convert_to_tensor=False).tolist()],
        n_results=50
    )
    dense_results = dense_response['ids'][0]

    rrf_ranking = reciprocal_rank_fusion(dense_results, sparse_results)
    retrieved_ids = [doc_id for doc_id, score in rrf_ranking[:10]]

    # Đo lường
    retrieval_results["Recall@1"].append(recall_at_k(retrieved_ids, relevant, 1))
    retrieval_results["Recall@3"].append(recall_at_k(retrieved_ids, relevant, 3))
    retrieval_results["Recall@5"].append(recall_at_k(retrieved_ids, relevant, 5))
    retrieval_results["MRR"].append(mean_reciprocal_rank(retrieved_ids, relevant))

# Tính trung bình các metrics
final_retrieval_metrics = {
    metric: round(sum(scores) / len(scores), 3) if scores else 0.0
    for metric, scores in retrieval_results.items()
}

print("Kết quả đo lường Retrieval (Hybrid BM25 + BGE-M3):")
for metric, score in final_retrieval_metrics.items():
    print(f"- {metric}: {score}")

# ==========================================
# LƯU KẾT QUẢ VÀO GOOGLE DRIVE
# ==========================================
eval_output_file = os.path.join(OUTPUT_DIR, "evaluation_metrics_report.json")

report_data = {
    "router_evaluation": router_report,
    "retrieval_evaluation": final_retrieval_metrics
}

with open(eval_output_file, "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=4)

print("\n" + "="*60)
print(f"Đã lưu báo cáo đánh giá thành công tại:\n{eval_output_file}")
print("="*60)

BẮT ĐẦU ĐÁNH GIÁ HỆ THỐNG (EVALUATION FRAMEWORK)

[1] ĐANG ĐÁNH GIÁ CẤP 1 - INTENT ROUTER...


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 20.706721094s.

In [10]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [11]:
import torch
from transformers import pipeline

print("Đang tải mô hình Llama-3-8B-Instruct (4-bit)...")
# Khởi tạo pipeline sinh văn bản chạy trên GPU
local_llm = pipeline(
    "text-generation",
    model="unsloth/llama-3-8b-Instruct-bnb-4bit",
    model_kwargs={"torch_dtype": torch.float16},
    device_map="auto",
)
print("Tải mô hình thành công! GPU đã sẵn sàng.")

Đang tải mô hình Llama-3-8B-Instruct (4-bit)...


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Tải mô hình thành công! GPU đã sẵn sàng.


In [12]:
import os
import json
from sklearn.metrics import classification_report
import numpy as np

# 1. HÀM ROUTER MỚI (DÙNG LOCAL LLM)
def route_query_local(query):
    """Sử dụng Local Llama-3 để phân loại ý định câu hỏi"""
    messages = [
        {"role": "system", "content": "Bạn là bộ định tuyến. Phân loại câu hỏi vào ĐÚNG 1 TRONG 4 loại: LOOKUP, COMPARE, RECOMMEND, EXPLAIN. Chỉ trả về 1 từ duy nhất."},
        {"role": "user", "content": f'Câu hỏi: "{query}"\nChỉ trả về 1 từ duy nhất: LOOKUP, COMPARE, RECOMMEND, hoặc EXPLAIN.'}
    ]

    prompt = local_llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = local_llm(prompt, max_new_tokens=10, do_sample=False, return_full_text=False)
    intent = outputs[0]["generated_text"].strip().upper()

    valid_intents = ["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"]
    for valid_intent in valid_intents:
        if valid_intent in intent: return valid_intent
    return "LOOKUP"

# 2. BẮT ĐẦU ĐÁNH GIÁ CẤP 1 (ROUTER)
print("="*60)
print("[1] ĐANG ĐÁNH GIÁ CẤP 1 - INTENT ROUTER (Dùng Local Llama-3)...")

router_test_data = [
    ("Cơm trắng 100g có bao nhiêu calo?", "LOOKUP"),
    ("Hàm lượng vitamin C trong cam là bao nhiêu?", "LOOKUP"),
    ("Thịt bò và thịt gà loại nào nhiều protein hơn?", "COMPARE"),
    ("Sữa tươi nguyên kem so với sữa ít béo khác gì?", "COMPARE"),
    ("Tôi muốn giảm cân, ăn gì cho bữa sáng?", "RECOMMEND"),
    ("Thực đơn 7 ngày cho người tiểu đường type 2", "RECOMMEND"),
    ("Chỉ số GI là gì và ảnh hưởng thế nào đến đường huyết?", "EXPLAIN"),
    ("Tại sao thiếu sắt lại gây thiếu máu?", "EXPLAIN")
]

y_true = []
y_pred = []

for query, true_intent in router_test_data:
    pred_intent = route_query_local(query)
    y_true.append(true_intent)
    y_pred.append(pred_intent)

router_report = classification_report(y_true, y_pred, labels=["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"], output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, labels=["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"], zero_division=0))

# 3. BẮT ĐẦU ĐÁNH GIÁ CẤP 2 (RETRIEVAL)
print("\n[2] ĐANG ĐÁNH GIÁ CẤP 2 - HYBRID RETRIEVAL...")

def get_id_by_keyword(keyword, r_type="food"):
    for r in all_records:
        if r["record_type"] == r_type and keyword.lower() in str(r.get("food_name_en", "")).lower():
            return r["record_id"]
    return "unknown"

retrieval_test_data = [
    {"query": "Thịt bò (Beef, grade I) chứa bao nhiêu protein?", "relevant_ids": [get_id_by_keyword("Beef, grade I")]},
    {"query": "Lượng Vitamin C cho trẻ em 4-6 tuổi", "relevant_ids": [r["record_id"] for r in all_records if r["record_type"] == "vitamin" and "4-6" in str(r.get("age", ""))]}
]

def recall_at_k(retrieved_ids, relevant_ids, k):
    return len(set(retrieved_ids[:k]) & set(relevant_ids)) / len(set(relevant_ids)) if relevant_ids else 0.0

def mean_reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in set(relevant_ids): return 1.0 / rank
    return 0.0

retrieval_results = {"Recall@1": [], "Recall@3": [], "Recall@5": [], "MRR": []}

for test_case in retrieval_test_data:
    query = test_case["query"]
    relevant = test_case["relevant_ids"]

    sparse_scores = bm25.get_scores(query.lower().split())
    sparse_results = [ids[i] for i in np.argsort(sparse_scores)[::-1][:50]]

    dense_results = collection.query(query_embeddings=[model.encode(query, convert_to_tensor=False).tolist()], n_results=50)['ids'][0]

    rrf_ranking = reciprocal_rank_fusion(dense_results, sparse_results)
    retrieved_ids = [doc_id for doc_id, score in rrf_ranking[:10]]

    retrieval_results["Recall@1"].append(recall_at_k(retrieved_ids, relevant, 1))
    retrieval_results["Recall@3"].append(recall_at_k(retrieved_ids, relevant, 3))
    retrieval_results["Recall@5"].append(recall_at_k(retrieved_ids, relevant, 5))
    retrieval_results["MRR"].append(mean_reciprocal_rank(retrieved_ids, relevant))

final_retrieval_metrics = {m: round(sum(s)/len(s), 3) if s else 0.0 for m, s in retrieval_results.items()}

print("Kết quả đo lường Retrieval:")
for metric, score in final_retrieval_metrics.items():
    print(f"- {metric}: {score}")

# 4. LƯU KẾT QUẢ VÀO DRIVE
OUTPUT_DIR = "/content/drive/MyDrive/nutrition_rag/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
eval_output_file = os.path.join(OUTPUT_DIR, "evaluation_metrics_report.json")

with open(eval_output_file, "w", encoding="utf-8") as f:
    json.dump({"router_evaluation": router_report, "retrieval_evaluation": final_retrieval_metrics}, f, ensure_ascii=False, indent=4)

print("\n" + "="*60)
print(f"Đã lưu báo cáo đánh giá thành công tại:\n{eval_output_file}")
print("="*60)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] ĐANG ĐÁNH GIÁ CẤP 1 - INTENT ROUTER (Dùng Local Llama-3)...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=819

              precision    recall  f1-score   support

      LOOKUP       0.67      1.00      0.80         2
     COMPARE       1.00      1.00      1.00         2
   RECOMMEND       1.00      1.00      1.00         2
     EXPLAIN       1.00      0.50      0.67         2

    accuracy                           0.88         8
   macro avg       0.92      0.88      0.87         8
weighted avg       0.92      0.88      0.87         8


[2] ĐANG ĐÁNH GIÁ CẤP 2 - HYBRID RETRIEVAL...
Kết quả đo lường Retrieval:
- Recall@1: 1.0
- Recall@3: 1.0
- Recall@5: 1.0
- MRR: 1.0

Đã lưu báo cáo đánh giá thành công tại:
/content/drive/MyDrive/nutrition_rag/outputs/evaluation_metrics_report.json


In [15]:
import re
import json
import os

print("="*60)
print("[3] ĐANG ĐÁNH GIÁ CẤP 3 - GENERATION (EXACT MATCH CẢI TIẾN)...")
print("="*60)

# 1. Hàm trích xuất Regex THÔNG MINH HƠN (Linh hoạt với dấu phẩy và từ đệm)
def extract_nutrition_values(text: str) -> dict:
    patterns = {
        "kcal":    r'(\d+(?:[.,]\d+)?)\s*(?:kcal|calo|Calo)',
        "protein": r'(\d+(?:[.,]\d+)?)\s*(?:g|gam|gram)\b.*?(?:protein|đạm)',
        "carbs":   r'(\d+(?:[.,]\d+)?)\s*(?:g|gam|gram)\b.*?(?:carb|tinh bột|bột đường)',
        "fat":     r'(\d+(?:[.,]\d+)?)\s*(?:g|gam|gram)\b.*?(?:chất béo|lipid|béo|fat)',
        "fiber":   r'(\d+(?:[.,]\d+)?)\s*(?:g|gam|gram)\b.*?(?:chất xơ|xơ|fiber)',
    }
    extracted = {}
    for nutrient, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Xử lý dấu phẩy thập phân kiểu Việt Nam (3,8 -> 3.8)
            val_str = match.group(1).replace(',', '.')
            extracted[nutrient] = float(val_str)
    return extracted

def numeric_exact_match(pred: str, gold: str, tolerance: float = 0.05) -> dict:
    """So sánh số liệu với sai số cho phép 5%"""
    pred_values = extract_nutrition_values(pred)
    gold_values = extract_nutrition_values(gold)

    results = {}
    for nutrient, gold_val in gold_values.items():
        if nutrient in pred_values:
            pred_val = pred_values[nutrient]
            if gold_val == 0:
                results[nutrient] = (pred_val == 0)
            else:
                results[nutrient] = abs(pred_val - gold_val) / gold_val <= tolerance
        else:
            results[nutrient] = False
    return results

# 2. Hàm RAG sử dụng Local Llama-3 để trả lời
def local_rag_answer(query):
    context = hybrid_search(query, top_k=3)
    if not context: return ""

    messages = [
        {"role": "system", "content": "Bạn là chuyên gia dinh dưỡng. Dựa vào [CONTEXT], trả lời ngắn gọn. Bắt buộc kèm số liệu và đơn vị (kcal, g)."},
        {"role": "user", "content": f"[CONTEXT]\n{context}\n\nCâu hỏi: {query}"}
    ]

    prompt = local_llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = local_llm(prompt, max_new_tokens=150, do_sample=False, return_full_text=False)
    return outputs[0]["generated_text"].strip()

# 3. Tập test đánh giá Exact Match
generation_test_data = [
    {
        "query": "Cơm trắng (Glutinous rice, milled) chứa bao nhiêu calo và đạm?",
        "gold_answer": "344 kcal, 8.6 g protein"
    },
    {
        "query": "Thịt bò (Beef, grade I) có hàm lượng chất béo (fat) là bao nhiêu?",
        "gold_answer": "3,8 g chất béo"
    }
]

total_nutrients = 0
correct_nutrients = 0

for idx, test in enumerate(generation_test_data):
    print(f"\n--- Test {idx + 1} ---")
    print(f"Câu hỏi: {test['query']}")

    pred_answer = local_rag_answer(test['query'])
    print(f"RAG Trả lời:\n{pred_answer}")

    match_results = numeric_exact_match(pred_answer, test['gold_answer'], tolerance=0.05)

    print("Exact Match Score:")
    for nutrient, is_correct in match_results.items():
        status = "✅ Pass" if is_correct else "❌ Fail"
        print(f" - {nutrient}: {status}")
        total_nutrients += 1
        if is_correct: correct_nutrients += 1

final_em_score = correct_nutrients / total_nutrients if total_nutrients > 0 else 0
print(f"\n=> TỔNG ĐIỂM EXACT MATCH (Numeric): {final_em_score * 100:.2f}%")

# Lưu bổ sung vào file report
eval_output_file = "/content/drive/MyDrive/nutrition_rag/outputs/evaluation_metrics_report.json"
try:
    with open(eval_output_file, "r", encoding="utf-8") as f:
        report_data = json.load(f)
except FileNotFoundError:
    report_data = {}

report_data["generation_evaluation"] = {
    "exact_match_score": final_em_score
}

with open(eval_output_file, "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=4)
print("Đã cập nhật điểm Generation vào file evaluation_metrics_report.json")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[3] ĐANG ĐÁNH GIÁ CẤP 3 - GENERATION (EXACT MATCH CẢI TIẾN)...

--- Test 1 ---
Câu hỏi: Cơm trắng (Glutinous rice, milled) chứa bao nhiêu calo và đạm?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG Trả lời:
According to the provided data, the average values for Glutinous rice, milled are:

* Calories: (344.0 + 346.0) / 2 = 345.0 kcal
* Protein: (8.6 + 8.4) / 2 = 8.5 g

So, Glutinous rice, milled contains approximately 345 kcal and 8.5 g of protein.
Exact Match Score:
 - kcal: ✅ Pass
 - protein: ✅ Pass

--- Test 2 ---
Câu hỏi: Thịt bò (Beef, grade I) có hàm lượng chất béo (fat) là bao nhiêu?
RAG Trả lời:
Thịt bò (Beef, grade I) có hàm lượng chất béo (fat) là 3,8 g.
Exact Match Score:
 - fat: ❌ Fail

=> TỔNG ĐIỂM EXACT MATCH (Numeric): 66.67%
Đã cập nhật điểm Generation vào file evaluation_metrics_report.json
